In [1]:
# %pip install -r ../requirements.txt
# # !! be aware !!
# # FastPFOR compression library python binding
# # only compiles with gcc -> on windows, requires mingw compiler
# #   as per detail:
# #   https://github.com/fast-pack/FastPFOR?tab=readme-ov-file#software-requirements
# # testing so far is only done on linux
# # if there are issues on windows...
# #   it is planned to provide a
# #   precompiled dll interface to FastPFOR

In [2]:
import asammdf          # use asammdf to extract samples & timestamps
import numpy as np      # check compression OK using np.allclose
from io import BytesIO  # 
import sys
sys.path.append('../')
from mdfc import (
    MDFCompressor, MDFDecompressor
)

In [3]:
# parameters to generate sample MF4 file
example_mdf_params = dict(
    # random int/float data
    include_random_ints=False,
    include_random_floats=False,
    # non-random int/float data
    include_sine_waves=True,
    include_keepalives=True,
    # extent & grouping parameters
    time_s=3600,
    sample_intervals_ms=(1000,500,250,100),
    channels_per_interval=50,
    channels_per_group=4,

    # adding +/- 5% jitter to each time group
    #   5% of the time interval magnitude
    #   thats jitter of random between, +/-...
    #       (500 us, 250 us, 125 us, 50 us)
    jitter_time=0.05
)

In [4]:
# uncompressed MDF file
from sample_data.generate_sample_data import generate_sample_file
MDF_FIL = BytesIO()
generate_sample_file(
    MDF_FIL, 
    compression=False,
    **example_mdf_params
)
MDF_FIL.seek(0); pass

96 total groups are generated


In [5]:
# size of uncompressed MDF file in MB
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()
print(
    f'{uncompressed_mdf_total_size/1000/1000:.2f} '
    'MB Uncompressed MDF File'
)

48.02 MB Uncompressed MDF File


In [6]:
# comparison against using deflate, 
# (using asammdf parameter compression=1)
DEFLATE_MDF_FIL = BytesIO()
generate_sample_file(
    DEFLATE_MDF_FIL, 
    compression=2,
    **example_mdf_params
)
DEFLATE_MDF_FIL.seek(0); pass

96 total groups are generated


In [7]:
# size of deflated MDF file in MB
deflate_mdf_total_size = DEFLATE_MDF_FIL.__sizeof__()
print(
    f'{deflate_mdf_total_size/1000/1000:.2f} '
    'MB Deflate MDF File'
)

28.51 MB Deflate MDF File


In [8]:
# ratio of deflate vs uncompressed
print(
    f'{uncompressed_mdf_total_size/deflate_mdf_total_size:.3f} '
    'CR using Deflate (MDF Standard)'
)

1.684 CR using Deflate (MDF Standard)


In [9]:
# configurable parameters for mdfc compression,
# which is just for lossy float compression
# for lossless fp compression, set:
#   tolerance, significands, minimum_tolerance
#   = -1  (the default values)
#   TODO allow some false-y value also, or None
#        but presently, it would raise value error :(
# in this example we can use these lossy params:
lossy_fp_params = dict(
    significands = 3,
    # ^ meaning: 
    #   3 additional digits
    #   after the significance
    #   of the smallest value
    #       uniquely for each channel
    #   eg: 
    #       if channel A min_value == 1e-5,
    #       that channel tolerance =  1e-8
    minimum_tolerance = 1e-3,
    # ^ meaning:
    #   minimum tolerance for all channels
)

In [10]:
# test params
DO_TEST_COMPRESSION   = True
DO_TEST_DECOMPRESSION = True

In [11]:
# %%timeit
# test compression
if DO_TEST_COMPRESSION:
    MDFC_FIL = BytesIO()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFCompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        mdfc_fil.compress_all_signals(
            mdf_fil,
            on_error='warn',
            # applies_zlib=True,
            **lossy_fp_params,
        )
        # presently, must call finish function,
        #   TODO it should be done on a (successful?) __exit__
        mdfc_fil.finish()
        # testing :)
        import copy
        times_axis = copy.deepcopy(mdfc_fil.time_axis)
        # parameters
        times_md = copy.deepcopy(mdfc_fil.time_metadata)
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

In [ ]:
mdfc_total_size = MDFC_FIL.__sizeof__()
print(
    f'{mdfc_total_size/1000/1000:.2f} '
    'MB MDFC File'
)

13.36 MB MDFC File


In [ ]:
# ratio of mdfc vs uncompressed
cr_vs_uncomp = (MDFC_FIL.__sizeof__() / MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs uncompressed is '
    f'{cr_vs_uncomp:.3f}, or {1/cr_vs_uncomp:.2f}x'
)

In [ ]:
# ratio of mdfc vs deflate
cr_vs_deflate = (MDFC_FIL.__sizeof__() / DEFLATE_MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs Deflate is '
    f'{cr_vs_deflate:.3f}, or {1/cr_vs_deflate:.2f}x'
)

In [ ]:
# %%timeit
# execute decompression & compare against original
def decompress_and_compare(sn, mdfc_fil, mdf_fil):
    # decompress the signal from mdfc
    # and compare it against the signal in mdf
    original_sig = mdf_fil.select([sn], raw=True)[0]
    original_timestamps = original_sig.timestamps
    original_samples = original_sig.samples
    
    # decompress mdfc signal
    res = mdfc_fil.decompress_signal(sn)

    # assert all close timestamps and values
    # timestamps... may have some minor losses
    #   due to float->scaleup->int on compression
    #   i think it should be understood that the retention
    #   should be based on the scale
    # print()
    #   which in this case is 0.05/1000
    # print(original_timestamps)
    # print(res.timestamps)
    assert np.allclose(
        original_timestamps,
        res.timestamps,
        # the "scaleup" applied on compression
        #   fp inaccuracies may be expected
        #   past this precision
        atol=(1/mdfc_fil.time_metadata[-1][0][1])
    ), f"{sn} timestamps not allclose!? :("
    assert np.allclose(
        original_samples,
        res.samples,
        # tolerance specification for float case
        # TODO perhaps this should be derived
        #   from compression metadata,
        #   ie the tolerance value used
        atol=lossy_fp_params['minimum_tolerance']
    ), f"{sn} samples not allclose!? :("


if DO_TEST_DECOMPRESSION:
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        # debugging
        md = mdfc_fil.metadata
        try:
            # test signal decompression
            for sn in mdf_fil.channels_db.keys():
                if sn == 'time': continue  #
                decompress_and_compare(sn, mdfc_fil, mdf_fil)
        except KeyError:
            print(f'{sn} found in MDF but not in compressed file...')
            # raise  # ?
        else:
            print("All signals have passed decompression check :)")
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

In [ ]:
# pass validity check :)

In [ ]:
# lets do some time checks...
# test_names = [
#     ... specific signal names...
# ]
test_names = None  # all signals

In [ ]:
%%timeit
# testing the speed of reading MDF (without compression)
MDF_FIL.seek(0)
with (
    asammdf.MDF(MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

In [ ]:
%%timeit
# testing the speed of reading MDF (with deflate compression)
DEFLATE_MDF_FIL.seek(0)
with (
    asammdf.MDF(DEFLATE_MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

In [ ]:
%%timeit
# testing the speed of reading the MDFC compressed file
MDFC_FIL.seek(0)
with MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil:
    if test_names is None:
        sigs = dfil.metadata.keys()
    else:
        sigs = test_names
    for sn in sigs:
        res = dfil.decompress_signal(sn)

In [ ]:
# end time checks :)